In [ ]:
!pip install captum transformers datasets

In [ ]:
import torch
import pandas as pd
import numpy as np

from transformers import DistilBertTokenizerFast
from transformers import DistilBertForSequenceClassification

from captum.attr import LayerIntegratedGradients
from captum.attr import visualization

import matplotlib.pyplot as plt

In [ ]:
!unzip tier_c_model.zip

Archive:  tier_c_model.zip
replace tokenizer_config.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.2 MB/s eta 0:00:00


In [ ]:
from transformers import DistilBertForSequenceClassification
from peft import PeftModel

base_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model = PeftModel.from_pretrained(base_model, ".")
model.eval()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): DistilBertSelfAttention(
                (q_lin): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=8, bias=False)
                  )
      

In [ ]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained(".")

In [ ]:
!unzip datasets.zip -d datasets

Archive:  datasets.zip
   creating: datasets/datasets/
  inflating: datasets/datasets/dataset_Class3_Class_Society_and_Social_Evolution_Charles_Dickens.xls  
  inflating: datasets/datasets/dataset_Class3_Crime_and_Justice_Charles_Dickens.xls  
  inflating: datasets/datasets/dataset_Class3_Human_Nature_and_Morality_Charles_Dickens.xls  
  inflating: datasets/datasets/dataset_Class3_Imperialism_and_Colonialism_Charles_Dickens.xls  
  inflating: datasets/datasets/dataset_Class3_Scientific_Discovery_and_Technology_Charles_Dickens.xls  
  inflating: datasets/datasets/dataset_Class_Society_and_Social_Evolution.xls  
  inflating: datasets/datasets/dataset_Crime_and_Justice.xls  
  inflating: datasets/datasets/dataset_Human_Class_Society_and_Social_Evolution.xls  
  inflating: datasets/datasets/dataset_Human_Crime_and_Justice.xls  
  inflating: datasets/datasets/dataset_Human_Human_Nature_and_Morality.xls  
  inflating: datasets/datasets/dataset_Human_Imperialism_and_Colonialism.xls  
  inflat

In [ ]:
text = """
Artificial intelligence has transformed the landscape of scientific discovery,
offering researchers unprecedented opportunities to accelerate innovation.
Through collaboration and careful experimentation, society continues to
benefit from these technological advancements.
"""

In [ ]:
import torch

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

with torch.no_grad():
    outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

print("Prediction:", prediction)

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Prediction: 1


In [ ]:
for name, module in model.named_modules():
    if "embed" in name.lower():
        print(name)

base_model.model.distilbert.embeddings
base_model.model.distilbert.embeddings.word_embeddings
base_model.model.distilbert.embeddings.position_embeddings
base_model.model.distilbert.embeddings.LayerNorm
base_model.model.distilbert.embeddings.dropout
base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_embedding_A
base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_embedding_B
base_model.model.distilbert.transformer.layer.0.attention.v_lin.lora_embedding_A
base_model.model.distilbert.transformer.layer.0.attention.v_lin.lora_embedding_B
base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_embedding_A
base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_embedding_B
base_model.model.distilbert.transformer.layer.1.attention.v_lin.lora_embedding_A
base_model.model.distilbert.transformer.layer.1.attention.v_lin.lora_embedding_B
base_model.model.distilbert.transformer.layer.2.attention.q_lin.lora_embedding_A
base_model.model.disti

In [ ]:
def forward_func(input_ids, attention_mask):
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )
    return outputs.logits[:,1]

In [ ]:
from captum.attr import LayerIntegratedGradients

lig = LayerIntegratedGradients(
    forward_func,
    model.base_model.model.distilbert.embeddings.word_embeddings
)

In [ ]:
inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

In [ ]:
attributions, delta = lig.attribute(
    inputs["input_ids"],
    additional_forward_args=(inputs["attention_mask"],),
    return_convergence_delta=True
)

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

In [ ]:
scores = attributions.sum(dim=-1).squeeze().detach().numpy()

In [ ]:
from captum.attr import visualization

visualization.visualize_text([
    visualization.VisualizationDataRecord(
        scores,
        0,
        1,
        "AI",
        "AI",
        0,
        tokens,
        delta
    )
])

In [ ]:
import pandas as pd
import torch

In [ ]:
import os

for file in os.listdir("datasets/datasets"):
    print(file)

dataset_Class_Society_and_Social_Evolution.xls
dataset_Class3_Human_Nature_and_Morality_Charles_Dickens.xls
dataset_Human_Class_Society_and_Social_Evolution.xls
dataset_Class3_Class_Society_and_Social_Evolution_Charles_Dickens.xls
dataset_Human_Human_Nature_and_Morality.xls
dataset_Human_Imperialism_and_Colonialism.xls
dataset_Scientific_Discovery_and_Technology.xls
dataset_Class3_Scientific_Discovery_and_Technology_Charles_Dickens.xls
dataset_Human_Scientific_Discovery_and_Technology.xls
dataset_Human_Crime_and_Justice.xls
dataset_Human_Nature_and_Morality.xls
dataset_Class3_Crime_and_Justice_Charles_Dickens.xls
dataset_Crime_and_Justice.xls
dataset_Class3_Imperialism_and_Colonialism_Charles_Dickens.xls
dataset_Imperialism_and_Colonialism.xls


In [ ]:
!pip install openpyxl

In [ ]:
file_path = "datasets/datasets/dataset_Human_Scientific_Discovery_and_Technology.xls"

with open(file_path, "rb") as f:
    print(f.read(20))

b'Topic,Paragraph,Clas'


In [ ]:
df = pd.read_csv(
    "datasets/datasets/dataset_Human_Scientific_Discovery_and_Technology.xls"
)

In [ ]:
print(df.head())
print(df.columns)

                                 Topic  \
0  Scientific Discovery and Technology   
1  Scientific Discovery and Technology   
2  Scientific Discovery and Technology   
3  Scientific Discovery and Technology   
4  Scientific Discovery and Technology   

                                           Paragraph  Class  
0  It appeared to have been fitted up as a chemic...  Human  
1  I had no opportunity to tell the baronet what ...  Human  
2  We were fairly after her now. The furnaces roa...  Human  
3  “The moor is very sparsely inhabited, and thos...  Human  
4  I had no opportunity to tell the baronet what ...  Human  
Index(['Topic', 'Paragraph', 'Class'], dtype='object')


In [ ]:
human_as_ai = []

for i in range(len(df)):
    text = str(df.iloc[i]["Paragraph"])

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()

    if pred == 1:          # predicted AI
        human_as_ai.append((i, text))

    if len(human_as_ai) == 3:
        break

print("Found", len(human_as_ai), "samples")

Found 3 samples


In [ ]:
for idx, text in human_as_ai:
    print("="*100)
    print("Sample", idx)
    print(text)
    print()

Sample 2
We were fairly after her now. The furnaces roared, and the powerful
engines whizzed and clanked, like a great metallic heart. Her sharp,
steep prow cut through the river-water and sent two rolling waves to
right and to left of us. With every throb of the engines we sprang and
quivered like a living thing. One great yellow lantern in our bows
threw a long, flickering funnel of light in front of us. Right ahead a
dark blur upon the water showed where the _Aurora_ lay, and the swirl
of white foam behind her spoke of the pace at which she was going. We
flashed past barges, steamers, merchant-vessels, in and out, behind
this one and round the other. Voices hailed us out of the darkness, but
still the _Aurora_ thundered on, and still we followed close upon her
track.

Sample 22
We were fairly after her now. The furnaces roared, and the powerful
engines whizzed and clanked, like a great metallic heart. Her sharp,
steep prow cut through the river-water and sent two rolling waves to
ri

In [ ]:
import pandas as pd
import torch

# Load your HUMAN dataset
df = pd.read_csv("datasets/datasets/dataset_Human_Scientific_Discovery_and_Technology.xls")

false_positives = []

for i in range(len(df)):

    text = str(df.loc[i, "Paragraph"])

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    # Human predicted as AI
    if prediction == 1:
        false_positives.append((i, text))

    if len(false_positives) == 3:
        break

print("False Positives Found:", len(false_positives))

False Positives Found: 3


In [ ]:
for idx, text in false_positives:

    print("="*100)
    print("Sample:", idx)
    print(text)
    print()

Sample: 2
We were fairly after her now. The furnaces roared, and the powerful
engines whizzed and clanked, like a great metallic heart. Her sharp,
steep prow cut through the river-water and sent two rolling waves to
right and to left of us. With every throb of the engines we sprang and
quivered like a living thing. One great yellow lantern in our bows
threw a long, flickering funnel of light in front of us. Right ahead a
dark blur upon the water showed where the _Aurora_ lay, and the swirl
of white foam behind her spoke of the pace at which she was going. We
flashed past barges, steamers, merchant-vessels, in and out, behind
this one and round the other. Voices hailed us out of the darkness, but
still the _Aurora_ thundered on, and still we followed close upon her
track.

Sample: 22
We were fairly after her now. The furnaces roared, and the powerful
engines whizzed and clanked, like a great metallic heart. Her sharp,
steep prow cut through the river-water and sent two rolling waves to
